In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [ ]:
# uncomment to use hosted db
os.environ["BIRDDOG_NOCODB_ENV"] = "CLOUD_PROD"
print(f"using {os.environ.get('BIRDDOG_NOCODB_ENV', 'LOCAL')} nocodb")


In [3]:
from birddog.database import Database
from birddog.database_updater import normalize_url

2026-08-15 09:06:26,637 [INFO] Using aws nocodb api: http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com
2026-08-15 09:06:26,773 [INFO] Translation is enabled. Using GCP translator
2026-08-15 09:06:26,774 [INFO] Using Google Cloud translation API
2026-08-15 09:06:26,774 [INFO] GoogleCloudTranslator using REST API


In [4]:
db = Database()

2026-08-15 09:06:26,947 [INFO] creating NocoDBDatabase(host=http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com, base_id=prsjtz30iuhk88f) instance
2026-08-15 09:06:27,141 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   20.00     5.21    39.00       0.00           24


In [ ]:
all_doc_ids = db.get_all_ids("Documents")

In [ ]:
len(all_doc_ids)

In [ ]:
doc_recs = db.read("Documents", all_doc_ids, fields=["url", "sha1_hash"])

In [ ]:
len(doc_recs)

In [ ]:
doc_recs[0]

In [ ]:
def group_equiv_records(recs):
    result = {}
    for rec in recs:
        url = rec.get("url")
        n_url = normalize_url(url)
        entries = result.get(n_url, [])
        entries.append(rec)
        result[n_url] = entries
    return result

In [ ]:
grouped_doc_recs = group_equiv_records(doc_recs)

In [ ]:
def select_aliased_doc_recs(groups):
    return {k: v for k, v in groups.items() if len(v) > 1}

In [ ]:
aliased_doc_recs = select_aliased_doc_recs(grouped_doc_recs)

In [ ]:
len(aliased_doc_recs)

In [ ]:
list(aliased_doc_recs.items())[0]

In [ ]:
def field_list(db, table_name="Documents"):
    recs = db.scan_all(
        "Schema", 
        where=("table_name", "eq", table_name), 
        fields="field_name")
    return [rec["field_name"] for rec in recs]

In [ ]:
doc_fields = field_list(db)

In [ ]:
def fold_field(recs, primary_rec, field_name):
    value = primary_rec.get(field_name)
    if value:
        return value
    for r in recs:
        value = r.get(field_name)
        if value:
            return value
    return primary_rec.get(field_name)    

In [ ]:
def fold_aliased_records(db, recs, fields=doc_fields):
    if len(recs) < 2:
        return recs[:1]
    n_url = normalize_url(recs[0]["url"])
    ids = [r["Id"] for r in recs]
    full_recs = db.read("Documents", ids, fields=fields)
    for rec in full_recs:
        if rec.get("url") == n_url:
            primary_rec = rec
            break
    result = primary_rec
    for field in fields:
        primary_rec[field] = fold_field(full_recs, primary_rec, field)
    return primary_rec

In [ ]:
folded_records = [
    fold_aliased_records(db, v)
    for v in aliased_doc_recs.values()
]

In [ ]:
len(folded_records)

In [ ]:
w_ids = db.write("Documents", folded_records)

In [ ]:
all([a == b["Id"] for a, b in zip(w_ids, folded_records)])

In [ ]:
def merge_page_links(db, aliased_recs):
    owners = []
    primary_id = None
    for r in aliased_recs:
        if r["url"] == normalize_url(r["url"]):
            primary_id = r["Id"]
        owners.extend(db.get_links("Documents", "owning_pages", r["Id"]))
    owners = list(set(owners))
    db.create_links("Documents", "owning_pages", primary_id, owners)
    return primary_id, owners

In [ ]:
merge_page_links(db, list(aliased_doc_recs.values())[0])

In [ ]:
for item in aliased_doc_recs.values():
    result = merge_page_links(db, item)
    print(result)

In [ ]:
def non_primaries(recs):
    return [r["Id"] for r in recs
            if r["url"] != normalize_url(r["url"])]

In [ ]:
np_ids = []
for v in aliased_doc_recs.values():
    np_ids.extend(non_primaries(v))
np_ids = list(set(np_ids))

In [ ]:
len(np_ids)

In [ ]:
np_ids

In [ ]:
db.delete("Documents", np_ids)

In [ ]:
def group_records_by_hash(recs):
    result = {}
    for rec in recs:
        sha1 = rec.get("sha1_hash")
        if sha1:
            entries = result.get(sha1, [])
            entries.append(rec)
            result[sha1] = entries
    return result

In [ ]:
docs_by_hash = group_records_by_hash(doc_recs)

In [ ]:
len(docs_by_hash)

In [ ]:
dupe_docs_by_hash = {
    k: v for k, v in docs_by_hash.items()
    if len(v) > 1
}

In [ ]:
len(dupe_docs_by_hash)

In [ ]:
list(dupe_docs_by_hash.items())[:5]

In [ ]:
lengths = [len(v) for v in dupe_docs_by_hash.values()]

In [ ]:
len(lengths)

In [ ]:
h = {}
for l in lengths:
    n = h.get(l, 0)
    h[l] = n + 1
h

In [14]:
fix_recs = []
cursor = None
while True:
    print(cursor)
    batch, cursor = db.scan(
        "Documents",
        view_name="dupes checked",
        fields=["dupes_checked", "url"],
        limit=1000,
        cursor=cursor,
    )
    fix_recs.extend(batch)
    if not cursor:
        break

None
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
53000
54000
2026-08-15 09:12:15,404 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   29.00     1.03    39.00       0.00           24
55000
56000
57000
58000
59000
60000
61000
62000
63000
64000
65000
66000
67000
68000
69000
70000
71000
72000
73000
74000
75000


In [15]:
len(fix_recs)

75200

In [16]:
for r in fix_recs:
    r["dupes_checked"] = False

In [17]:
db.write("Documents", fix_recs)

2026-08-15 09:13:16,495 [INFO] creating Reserver(table_name=Documents)
2026-08-15 09:13:17,394 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   32.00     0.35    35.18       0.00           24
2026-08-15 09:14:17,536 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  nocodb.internal:api                   37.00     5.64    35.16       0.00           24
2026-08-15 09:15:10,570 [INFO] [1/5] Error: HTTPConnectionPool(host='nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com', port=80): Read timed out. (read timeout=10). Retrying in 1.27s...
2026-08-15 09:15:20,807 [INFO] 
service throttle report:
  host_key                    

[272613,
 272614,
 272615,
 272616,
 272617,
 272618,
 272619,
 272620,
 272621,
 272622,
 272623,
 272624,
 272625,
 272626,
 272627,
 272628,
 272629,
 272630,
 272631,
 272632,
 272633,
 272634,
 272635,
 272636,
 272637,
 272638,
 272639,
 272640,
 272641,
 272642,
 272643,
 272644,
 272645,
 272646,
 272647,
 272648,
 272649,
 272650,
 272651,
 272652,
 272653,
 272654,
 272655,
 272656,
 272665,
 272658,
 272672,
 272661,
 272662,
 272664,
 272676,
 272666,
 272677,
 272667,
 272679,
 272681,
 272685,
 272687,
 272688,
 272689,
 272690,
 272692,
 272700,
 272701,
 272702,
 272704,
 272705,
 272706,
 272707,
 272708,
 272711,
 272715,
 272716,
 272720,
 272724,
 272727,
 272728,
 272729,
 272730,
 272731,
 272733,
 272734,
 272739,
 272740,
 272741,
 272742,
 272743,
 272745,
 272746,
 272748,
 272749,
 272789,
 272751,
 272792,
 272753,
 272754,
 272794,
 272797,
 272798,
 272800,
 272763,
 272764,
 272767,
 272768,
 272769,
 272772,
 272773,
 272774,
 272776,
 272777,
 272778,
 